In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

train=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
test=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")


/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/sample_submission.csv
/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv
/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv


In [2]:
import torch
# GPU（CUDA）が利用可能かチェック
is_cuda = torch.cuda.is_available()
print(f"CUDA Available: {is_cuda}")

# 利用可能な場合、デバイス名を取得
if is_cuda:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
# GPUが使えれば 'cuda'、使えなければ 'cpu' を指定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CUDA Available: False
Using device: cpu


In [3]:
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler
import numpy as np

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

# Find common columns between train and test (excluding "DIC" from train)
common_columns = train.drop(columns=["DIC"]).columns.intersection(test.columns)

# Select the common columns for both train and test
X = train[common_columns]
y = train["DIC"]
test = test[common_columns]

# Replace NaNs with the column mean
X = X.fillna(X.mean())
test = test.fillna(test.mean())

# Normalize the input features
x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)
test_scaled = x_scaler.transform(test)

# Split the training data before fitting the target scaler
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2
 )


In [4]:
from torch.utils.data import Dataset, DataLoader, RandomSampler

class OceanChemistryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y.values, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

sample_ratio = 1.0

train_dataset = OceanChemistryDataset(X_train, y_train)
val_dataset = OceanChemistryDataset(X_val, y_val)

train_sampler = RandomSampler(train_dataset, replacement=True, num_samples=int(len(train_dataset) * sample_ratio))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, sampler=train_sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [5]:
class MLPModel(nn.Module):
    def __init__(self, input_size, dropout=0.0):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu1 = nn.ReLU()
        #self.dropout1 = nn.Dropout(p=dropout)
        self.fc2 = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        #self.dropout2 = nn.Dropout(p=dropout)
        self.fc3 = nn.Linear(8, 16)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(p=dropout)
        self.fc4 = nn.Linear(16, 128)
        self.relu4 = nn.ReLU()
        self.dropout4 = nn.Dropout(p=dropout)
        self.fc5 = nn.Linear(128, 16)
        self.relu5 = nn.ReLU()
        self.dropout5 = nn.Dropout(p=dropout)
        self.fc6 = nn.Linear(16, 64)
        self.relu6 = nn.ReLU()
        #self.dropout6 = nn.Dropout(p=dropout)
        self.fc7 = nn.Linear(64, 1)
        # Output layer for regression

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        #x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        #x = self.dropout2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        x = self.relu4(x)
        x = self.dropout4(x)
        x = self.fc5(x)
        x = self.relu5(x)
        x = self.dropout5(x)
        x = self.fc6(x)
        x = self.relu6(x)
        #x = self.dropout6(x)
        x = self.fc7(x)
        return x


model = MLPModel(input_size=X_train.shape[1], dropout=0.0015)


In [6]:
import torch.optim as optim
import copy

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=100,
    threshold=1e-4,
    min_lr=1e-4,
)


# Training function
def train_model(model, train_loader, val_loader, epochs=1500, patience=80):
    best_val_loss = float("inf")
    best_model_state = None
    patience_count = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze(-1)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch).squeeze(-1)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

        val_loss_average = val_loss / len(val_loader)
        scheduler.step(val_loss_average)

        if val_loss  < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        elif val_loss / len(val_loader)  < best_val_loss / len(val_loader) + 10:
            patience_count -= 10
        else:
            patience_count += 1

        if epoch % 100 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {running_loss / len(train_loader)}, Validation Loss: {val_loss_average}, Learning Rate: {current_lr}")

        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    print(f"Best Validation Loss: {best_val_loss/len(val_loader)}")
    model.load_state_dict(best_model_state)

# Train the model
model.to(device)
train_model(model, train_loader, val_loader, epochs=3000, patience=1000)


Epoch 1/3000, Train Loss: 2805918.6173986485, Validation Loss: 432422.525, Learning Rate: 0.01
Epoch 101/3000, Train Loss: 987.6093370592272, Validation Loss: 576.9892593383789, Learning Rate: 0.01
Epoch 201/3000, Train Loss: 265.9812197298617, Validation Loss: 135.66200103759766, Learning Rate: 0.01
Epoch 301/3000, Train Loss: 488.9391052658493, Validation Loss: 85.63390617370605, Learning Rate: 0.01
Epoch 401/3000, Train Loss: 501.72377921439505, Validation Loss: 102.265283203125, Learning Rate: 0.01
Epoch 501/3000, Train Loss: 162.4578762569943, Validation Loss: 207.10531768798828, Learning Rate: 0.01
Epoch 601/3000, Train Loss: 140.1592995411641, Validation Loss: 44.34900455474853, Learning Rate: 0.005
Epoch 701/3000, Train Loss: 132.01452381546432, Validation Loss: 34.165801429748534, Learning Rate: 0.0025
Epoch 801/3000, Train Loss: 137.74331788758974, Validation Loss: 31.454603004455567, Learning Rate: 0.0025
Epoch 901/3000, Train Loss: 118.51781102773306, Validation Loss: 25.71

In [7]:
# Convert the test set into a torch tensor
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

# Set the model to evaluation mode
model.eval()
model = model.to("cpu")

# Make predictions in standardized target space
with torch.no_grad():
    predictions = model(test_tensor).squeeze().numpy()

# Convert predictions back to the original DIC scale


# Prepare submission
submission = pd.DataFrame({"id": range(1455, 1455 + len(predictions)), "DIC": predictions})
submission.to_csv("submission.csv", index=False)
